# Prognoosiselgitused ja ajalooline kontekst

SHAP selgitab mudeli prognoosi. Prognoosivea diagnoos vajab lisaks tegelikku viga ja võrreldavat konteksti. Kõrge SHAP-väärtus ei ole tõend, et tunnus põhjustas vea.

In [ ]:
from pathlib import Path
from functools import partial
import sys
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "run_pipeline.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common import latest_run, read_json
from src.reporting import show_table, show_figure, show_run
from src.reporting import markdown_table

RUN = latest_run()
notes = read_json(RUN / "reports/interpretations.json")
table = partial(show_table, RUN, limit=30)
figure = partial(show_figure, RUN)
show_run(RUN)

In [ ]:
figure("figures/shap.png", notes["xai"])
pfi = pd.read_csv(RUN / "explanations/permutation_importance.csv")
summary = pfi.groupby("group").MAE_increase.agg(["mean", "std"]).reset_index()
display(
    Markdown(
        markdown_table(
            summary.rename(
                columns={"mean": "MAE suurenemise keskmine", "std": "Korduste standardhälve"}
            )
        )
    )
)
display(
    Markdown(
        "PFI tabel sisaldab üksiktunnuseid ja ühiseid tunnuserühmi. Suurem väärtus näitab selle mudeli tundlikkust permuteerimisele. Korduste standardhälve kirjeldab permutatsiooni varieeruvust, mitte üldistusvea usaldusvahemikku; korreleeritud tunnused võivad varjata üksteise rolli."
    )
)

In [ ]:
figure("figures/pfi.png", notes["pfi"])

In [ ]:
table(
    "explanations/matched_shap_comparison.csv",
    columns=[
        "feature",
        "pairs",
        "large_mean_abs_shap",
        "matched_small_mean_abs_shap",
        "difference",
    ],
    rename={
        "pairs": "Sobitatud paaride arv",
        "large_mean_abs_shap": "Suure vea keskmine absoluutpanus",
        "matched_small_mean_abs_shap": "Väikese vea sobitatud absoluutpanus",
        "difference": "Panuste keskmine erinevus",
    },
)
display(
    Markdown(
        "Võrdlus sobitab sama tunni ja nädalapäeva juhtumid teadaoleva koormuse ning temperatuuri järgi. Tulemused kirjeldavad mudeli kasutatud info erinevusi; kontrollide kordumine ja mõõtmata kontekst takistavad põhjuslikku tõlgendust. Puuduvate paaride puhul järeldusest loobutakse."
    )
)

In [ ]:
table(
    "explanations/stability.csv",
    rename={
        "sample_a": "Mudeli/valimi variant A",
        "sample_b": "Mudeli/valimi variant B",
        "top5_jaccard": "Viie olulisima tunnuse Jaccardi sarnasus",
    },
    limit=30,
)
display(
    Markdown(
        notes["xai"]
        + " Sama ankurvalimi kordussobituste võrdlus eristab mudelitundlikkust alamperioodide kontekstimuutusest; eri kvartalite võrdlus sisaldab mõlemat."
    )
)

In [ ]:
table(
    "explanations/historical_cases.csv",
    columns=[
        "query_id",
        "origin_time",
        "target_time",
        "prediction",
        "target_next_hour",
        "error",
        "distance",
    ],
    rename={
        "query_id": "Päringu tunnus",
        "origin_time": "Ajalooline prognoosihetk",
        "error": "Suunatud viga",
        "distance": "Standardiseeritud kaugus",
    },
    limit=10,
)
display(
    Markdown(
        "Ajaloolised analoogid on ainult päringuhetkeks teadaolevad valimivälised prognoosid. Lähedane tunnusruum ei taga sama toimemehhanismi. Esimesel diagnoosimishetkel ei pruugi varasemaid sobivaid juhtumeid olla; see piirab hüpoteesi tõendeid."
    )
)